<a href="https://colab.research.google.com/github/myla48/datascience/blob/main/Aquaculture_Pond_Detection_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aquaculture Pond Detection from Satellite Time-Series Data

This notebook provides an end-to-end solution for detecting aquaculture ponds using Sentinel-1 (SAR) and Sentinel-2 (optical) satellite imagery. It covers data loading, EDA, feature engineering, model training with cross-validation and domain shift simulation, evaluation, and submission file generation.

## 1. Setup and Imports

In [ ]:
# Install necessary libraries
!pip install pandas numpy scikit-learn lightgbm xgboost catboost
!pip install tqdm  # For progress bars

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
import re
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)

# Helper function to set seeds for models
def set_all_seeds(seed):
    np.random.seed(seed)
    # Add other library-specific seed settings if necessary (e.g., tensorflow, pytorch)

set_all_seeds(SEED)

## 2. Load Data

We will load the `Train.csv`, `Test.csv`, and `SampleSubmission.csv` files. The column names for the final submission will be inferred from `SampleSubmission.csv` to ensure exact matching.

In [ ]:
# Load datasets
train_df = pd.read_csv('/content/Train.csv')
test_df = pd.read_csv('/content/Test.csv')
sample_submission_df = pd.read_csv('/content/SampleSubmission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Sample Submission shape: {sample_submission_df.shape}")

# Infer submission column names
SUBMISSION_ID_COL = sample_submission_df.columns[0]
SUBMISSION_TARGET_COL = sample_submission_df.columns[1]
SUBMISSION_PROB_COL = sample_submission_df.columns[2]

print(f"Submission ID Column: {SUBMISSION_ID_COL}")
print(f"Submission Target Column: {SUBMISSION_TARGET_COL}")
print(f"Submission Probability Column: {SUBMISSION_PROB_COL}")

# Display first few rows of train data
print("\nTrain Data Head:")
display(train_df.head())

Train data shape: (1821, 146)
Test data shape: (1030, 145)
Sample Submission shape: (1030, 3)
Submission ID Column: ID
Submission Target Column: TargetF1
Submission Probability Column: TargetRAUC

Train Data Head:


,ID,label,VH_01,VV_01,blue_01,green_01,nir_01,nira_01,re1_01,re2_01,...,blue_12,green_12,nir_12,nira_12,re1_12,re2_12,re3_12,red_12,swir1_12,swir2_12
0,ID_TR_NEW_XVGKFMLNRJ,0,-29.099645,-22.471573,1665,1719,1367,1270,1689,1416,...,1639,1826,1395,1384,1736,1449,1436,1626,1307,1216
1,ID_TR_NEW_GP8KNSWVP6,0,-19.470574,-10.752340,1579,1740,2245,2231,2060,2147,...,1490,1622,2090,2109,1924,1941,2020,1779,2298,2131
2,ID_TR_NEW_87X3957MVS,1,-20.964854,-8.792675,1850,2345,3664,3470,2935,3384,...,1438,1673,1335,1289,1588,1377,1307,1503,1237,1168
3,ID_TR_NEW_T4JMRPKHS3,0,-18.728208,-4.689202,1279,1297,1713,1698,1527,1531,...,1405,1550,2454,2397,1894,2040,2179,1748,2574,1977
4,ID_TR_NEW_2CTUQQ8KLU,0,-29.630941,-23.370157,1625,1741,1339,1258,1658,1362,...,1582,1856,1365,1306,1627,1370,1367,1641,1219,1147


## 3. Exploratory Data Analysis (EDA) and Initial Preprocessing

In this section, we will:
- Replace the `-9999` missing value indicator with `np.nan`.
- Inspect column naming patterns.
- Analyze missing value rates.
- Check the class balance of the target variable in the training set.

**Note on Column Naming:** The feature columns are expected to follow a pattern like `BAND_monthX_BANDNAME` or `BANDNAME_monthX`. The regex below `r'_(m\d{{1,2}})_([A-Za-z0-9_]+)'` is designed to capture `month` and `band name` from patterns like `BANDNAME_mX` or `_mX_BANDNAME`. Please adjust the `FEATURE_COL_PATTERN` regex if your actual column names deviate significantly.

In [ ]:
# Identify feature columns
# Exclude ID and Target columns
ID_COL = 'ID'
TARGET_COL = 'label' # Corrected target column name based on train_df.head()

feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]

# Replace -9999 with NaN
def replace_missing_values(df, feature_cols, missing_val=-9999):
    df_copy = df.copy()
    for col in feature_cols:
        df_copy[col] = df_copy[col].replace(missing_val, np.nan)
    return df_copy

train_df_processed = replace_missing_values(train_df, feature_cols)
test_df_processed = replace_missing_values(test_df, feature_cols)

print("\nAfter replacing -9999 with NaN:")
print("Train data info:")
train_df_processed.info(verbose=False, show_counts=True)
print("\nTest data info:")
test_df_processed.info(verbose=False, show_counts=True)


# Parse column names to extract month and band information
# Example pattern: 'VH_01', 'B2_12' etc.
# The column names are like 'BAND_MM' (e.g., 'VH_01', 'blue_12')

# Corrected FEATURE_COL_PATTERN
FEATURE_COL_PATTERN = r'([A-Za-z0-9_]+)_(\d{2})' # Matches e.g. 'VH_01', 'B2_10'
# If the pattern is like '01_VH', use r'(\d{2})_([A-Za-z0-9_]+)'
# If a different pattern, please update FEATURE_COL_PATTERN

parsed_features = []
for col in feature_cols:
    match = re.match(FEATURE_COL_PATTERN, col)
    if match:
        band_name, month = match.groups()
        parsed_features.append({'column': col, 'band': band_name, 'month': int(month)})

if not parsed_features:
    print(f"\nWARNING: No features matched the pattern '{FEATURE_COL_PATTERN}'. Please adjust FEATURE_COL_PATTERN if needed.")
    print("Sample feature columns:", feature_cols[:10])
else:
    print(f"\nSuccessfully parsed {len(parsed_features)} feature columns using pattern '{FEATURE_COL_PATTERN}'.")
    print("Example parsed features:")
    for pf in parsed_features[:5]:
        print(f"  Column: {pf['column']}, Band: {pf['band']}, Month: {pf['month']}")

# Store mapping for later use
feature_mapping = pd.DataFrame(parsed_features)


# Check class balance in training data
class_counts = train_df_processed[TARGET_COL].value_counts(normalize=True)
print(f"\nClass balance in training data:\n{class_counts}")


After replacing -9999 with NaN:
Train data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1821 entries, 0 to 1820
Columns: 146 entries, ID to swir2_12
dtypes: float64(24), int64(121), object(1)
memory usage: 2.0+ MB

Test data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Columns: 145 entries, ID to swir2_12
dtypes: float64(144), object(1)
memory usage: 1.1+ MB

Successfully parsed 144 feature columns using pattern '([A-Za-z0-9_]+)_(\d{2})'.
Example parsed features:
  Column: VH_01, Band: VH, Month: 1
  Column: VV_01, Band: VV, Month: 1
  Column: blue_01, Band: blue, Month: 1
  Column: green_01, Band: green, Month: 1
  Column: nir_01, Band: nir, Month: 1

Class balance in training data:
label
0    0.596376
1    0.403624
Name: proportion, dtype: float64


### Define Band Groups and Utility Functions

To facilitate feature engineering, we'll define the SAR and optical bands and create a utility function to extract monthly data for each row.

In [ ]:
# Define band groups
SAR_BANDS = ['VH', 'VV']
OPTICAL_BANDS = ['blue', 'green', 'red', 're1', 're2', 're3', 'nir', 'nira', 'swir1', 'swir2'] # Assuming 're1', 're2', 're3' for red-edge
ALL_BANDS = SAR_BANDS + OPTICAL_BANDS

# Dynamically extract all unique band names from feature_mapping (excluding month part)
unique_bands = feature_mapping['band'].unique().tolist()
print(f"Detected unique bands from data: {unique_bands}")

# Helper function to get all monthly columns for a given band
def get_monthly_columns_for_band(df, band_name):
    # Updated regex to match BAND_MM pattern (e.g., 'VH_01')
    band_cols = [col for col in df.columns if re.match(f'{re.escape(band_name)}_(\d{{2}})', col, re.IGNORECASE) ]
    # Sort columns by month number to ensure temporal order, with a safeguard for re.search returning None
    band_cols.sort(key=lambda x: int(re.search(r'_(\d+)', x).group(1)) if re.search(r'_(\\d+)', x) else 999)
    return band_cols

# Create a dictionary to store monthly column names for each band
MONTHLY_BAND_COLS = {band: get_monthly_columns_for_band(train_df, band) for band in unique_bands}

print("\nMonthly column mapping examples:")
for band, cols in list(MONTHLY_BAND_COLS.items())[:3]:
    print(f"  {band}: {cols[:5]}...")

Detected unique bands from data: ['VH', 'VV', 'blue', 'green', 'nir', 'nira', 're1', 're2', 're3', 'red', 'swir1', 'swir2']

Monthly column mapping examples:
  VH: ['VH_01', 'VH_02', 'VH_03', 'VH_04', 'VH_05']...
  VV: ['VV_01', 'VV_02', 'VV_03', 'VV_04', 'VV_05']...
  blue: ['blue_01', 'blue_02', 'blue_03', 'blue_04', 'blue_05']...


### 4. Feature Engineering

This section focuses on engineering features that are robust to missing data and the domain shift between train and test sets. We will compute per-band summary statistics over valid months, count valid months, and calculate spectral indices.

#### 4.1 Detect Valid Months

First, we need a robust way to identify which months are 'valid' for each row, considering that Sentinel-1 (SAR) data is always present if a month is valid, while Sentinel-2 (optical) data can be missing due to cloud cover.

In [ ]:
def identify_valid_months(df, sar_bands, optical_bands):
    """
    Identifies valid months for each row based on SAR band presence.
    Returns a DataFrame with boolean masks for SAR and optical data per month.
    """
    num_months = 12 # Assuming 12 months in the data
    valid_sar_months_mask = pd.DataFrame(index=df.index, columns=range(1, num_months + 1), dtype=bool)
    valid_optical_months_mask = pd.DataFrame(index=df.index, columns=range(1, num_months + 1), dtype=bool)

    # A month is 'valid' if at least one SAR band is not NaN for that month
    # and not all SAR bands are NaN (to avoid cases where a band might be missing but the month isn't totally masked)
    for month_idx in range(1, num_months + 1):
        sar_month_cols = [MONTHLY_BAND_COLS[band][month_idx-1] for band in sar_bands if band in MONTHLY_BAND_COLS and month_idx-1 < len(MONTHLY_BAND_COLS[band])]
        if sar_month_cols:
            # A month is valid if ANY of its SAR bands are not NaN
            valid_sar_months_mask[month_idx] = df[sar_month_cols].notna().any(axis=1)
        else:
            valid_sar_months_mask[month_idx] = False # No SAR data for this month

        # Optical data is valid if all its bands are not NaN for that month AND the month is sar-valid
        # This ensures we don't count optical data for months that are supposed to be fully masked.
        optical_month_cols = [MONTHLY_BAND_COLS[band][month_idx-1] for band in optical_bands if band in MONTHLY_BAND_COLS and month_idx-1 < len(MONTHLY_BAND_COLS[band])]
        if optical_month_cols:
            valid_optical_months_mask[month_idx] = df[optical_month_cols].notna().all(axis=1) & valid_sar_months_mask[month_idx]
        else:
            valid_optical_months_mask[month_idx] = False

    return valid_sar_months_mask, valid_optical_months_mask

# Apply to both train and test data
train_valid_sar_months, train_valid_optical_months = identify_valid_months(train_df_processed, SAR_BANDS, OPTICAL_BANDS)
test_valid_sar_months, test_valid_optical_months = identify_valid_months(test_df_processed, SAR_BANDS, OPTICAL_BANDS)

print("\nExample of valid SAR months for the first 5 train rows:")
display(train_valid_sar_months.head())
print("\nExample of valid optical months for the first 5 train rows:")
display(train_valid_optical_months.head())

# Calculate count of valid months per row
train_df_processed['num_valid_sar_months'] = train_valid_sar_months.sum(axis=1)
test_df_processed['num_valid_sar_months'] = test_valid_sar_months.sum(axis=1)

train_df_processed['num_valid_optical_months'] = train_valid_optical_months.sum(axis=1)
test_df_processed['num_valid_optical_months'] = test_valid_optical_months.sum(axis=1)

print("\nDistribution of valid SAR months in training set:")
print(train_df_processed['num_valid_sar_months'].value_counts().sort_index())
print("\nDistribution of valid SAR months in test set:")
print(test_df_processed['num_valid_sar_months'].value_counts().sort_index())





Example of valid SAR months for the first 5 train rows:


,1,2,3,4,5,6,7,8,9,10,11,12
0,True,True,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True,True,True



Example of valid optical months for the first 5 train rows:


,1,2,3,4,5,6,7,8,9,10,11,12
0,True,True,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True,True,True



Distribution of valid SAR months in training set:
num_valid_sar_months
12    1821
Name: count, dtype: int64

Distribution of valid SAR months in test set:
num_valid_sar_months
4    345
5    343
6    342
Name: count, dtype: int64


#### 4.2 Generate Monthly DataFrames

To simplify feature engineering, we will reshape the data into a monthly format, where each row represents a location at a specific month. This will make it easier to compute spectral indices and aggregate statistics.

In [ ]:
def melt_to_monthly_dataframe(df, valid_months_mask, bands, id_col, target_col=None):
    """
    Melts the DataFrame into a monthly format, keeping only valid months.
    """
    data_melted = []
    num_months = valid_months_mask.shape[1]

    for month_idx in range(1, num_months + 1):
        month_df = df.loc[valid_months_mask[month_idx]].copy() # Only include rows for valid months
        if month_df.empty: continue

        month_data = {'month': month_idx}
        month_data[id_col] = month_df[id_col]
        if target_col and target_col in month_df.columns:
            month_data[target_col] = month_df[target_col]

        for band in bands:
            # Corrected: Remove 'm' from the band_col_name construction
            band_col_name = f'{band}_{month_idx:02d}' # Construct the column name based on actual pattern
            if band_col_name in month_df.columns:
                month_data[band] = month_df[band_col_name]

        data_melted.append(pd.DataFrame(month_data))

    if not data_melted:
        return pd.DataFrame() # Return empty DataFrame if no valid data
    return pd.concat(data_melted, ignore_index=True)

print("Melting train data...")
train_monthly_df = melt_to_monthly_dataframe(train_df_processed, train_valid_sar_months, unique_bands, ID_COL, TARGET_COL)
print(f"Train monthly data shape: {train_monthly_df.shape}")

print("Melting test data...")
test_monthly_df = melt_to_monthly_dataframe(test_df_processed, test_valid_sar_months, unique_bands, ID_COL)
print(f"Test monthly data shape: {test_monthly_df.shape}")

print("\nTrain Monthly Data Head:")
display(train_monthly_df.head())

print("\nTest Monthly Data Head:")
display(test_monthly_df.head())

Melting train data...
Train monthly data shape: (21852, 15)
Melting test data...
Test monthly data shape: (5147, 14)

Train Monthly Data Head:


,month,ID,label,VH,VV,blue,green,nir,nira,re1,re2,re3,red,swir1,swir2
0,1,ID_TR_NEW_XVGKFMLNRJ,0,-29.099645,-22.471573,1665,1719,1367,1270,1689,1416,1379,1705,1154,1141
1,1,ID_TR_NEW_GP8KNSWVP6,0,-19.470574,-10.752340,1579,1740,2245,2231,2060,2147,2161,1968,2554,2422
2,1,ID_TR_NEW_87X3957MVS,1,-20.964854,-8.792675,1850,2345,3664,3470,2935,3384,3451,2032,3189,2397
3,1,ID_TR_NEW_T4JMRPKHS3,0,-18.728208,-4.689202,1279,1297,1713,1698,1527,1531,1590,1468,1669,1421
4,1,ID_TR_NEW_2CTUQQ8KLU,0,-29.630941,-23.370157,1625,1741,1339,1258,1658,1362,1383,1708,1158,1128



Test Monthly Data Head:


,month,ID,VH,VV,blue,green,nir,nira,re1,re2,re3,red,swir1,swir2
0,1,ID_TS_NEW_7SPRN3PB,-34.017096,-22.539108,1540.0,1693.0,1300.0,1321.0,1629.0,1412.0,1378.0,1580.0,1405.0,1397.0
1,1,ID_TS_NEW_D1E33N0Z,-28.777638,-17.958622,1491.0,1916.0,1383.0,1389.0,1935.0,1457.0,1528.0,1723.0,1340.0,1309.0
2,1,ID_TS_NEW_T2466L38,-29.316786,-21.411090,1800.0,1914.0,1439.0,1553.0,1791.0,1525.0,1571.0,1824.0,1527.0,1339.0
3,1,ID_TS_NEW_WZG3T1DJ,-30.207398,-19.692385,1041.0,1428.0,1111.0,1096.0,1604.0,1186.0,1198.0,1181.0,1163.0,1154.0
4,1,ID_TS_NEW_1MX99FFB,-31.397668,-17.780978,2000.0,2048.0,1878.0,1895.0,2099.0,1931.0,1931.0,1949.0,1736.0,1621.0


#### 4.3 Compute Spectral Indices

We will now calculate common spectral indices for both optical and SAR bands. These indices are derived from band combinations and are known to be sensitive to water and vegetation, which are key characteristics for detecting aquaculture ponds.

In [ ]:
def calculate_spectral_indices(df):
    """
    Calculates spectral indices for both optical and SAR bands.
    Assumes band names are columns in the dataframe (e.g., 'blue', 'green', 'red').
    """
    df_copy = df.copy()

    # Optical Indices
    # NDWI: (Green - NIR) / (Green + NIR) - sensitive to water content
    if 'green' in df_copy.columns and 'nir' in df_copy.columns:
        df_copy['NDWI'] = (df_copy['green'] - df_copy['nir']) / (df_copy['green'] + df_copy['nir'])

    # NDVI: (NIR - Red) / (NIR + Red) - sensitive to vegetation greenness
    if 'nir' in df_copy.columns and 'red' in df_copy.columns:
        df_copy['NDVI'] = (df_copy['nir'] - df_copy['red']) / (df_copy['nir'] + df_copy['red'])

    # MNDWI: (Green - SWIR1) / (Green + SWIR1) - modified NDWI, often better for water
    if 'green' in df_copy.columns and 'swir1' in df_copy.columns:
        df_copy['MNDWI'] = (df_copy['green'] - df_copy['swir1']) / (df_copy['green'] + df_copy['swir1'])

    # NDMI: (NIR - SWIR1) / (NIR + SWIR1) - sensitive to moisture content
    if 'nir' in df_copy.columns and 'swir1' in df_copy.columns:
        df_copy['NDMI'] = (df_copy['nir'] - df_copy['swir1']) / (df_copy['nir'] + df_copy['swir1'])

    # SAR Indices
    # VH/VV Ratio
    if 'VH' in df_copy.columns and 'VV' in df_copy.columns:
        df_copy['VH_VV_ratio'] = df_copy['VH'] / df_copy['VV']

    # VH-VV Difference
    if 'VH' in df_copy.columns and 'VV' in df_copy.columns:
        df_copy['VH_VV_diff'] = df_copy['VH'] - df_copy['VV']

    # Handle NaNs from division by zero or missing bands in index calculation
    # These NaNs will be handled during aggregation
    return df_copy

print("Calculating spectral indices for train monthly data...")
train_monthly_df_indices = calculate_spectral_indices(train_monthly_df)
print("Calculating spectral indices for test monthly data...")
test_monthly_df_indices = calculate_spectral_indices(test_monthly_df)

print("\nTrain Monthly Data with Indices Head:")
display(train_monthly_df_indices.head())


Calculating spectral indices for train monthly data...
Calculating spectral indices for test monthly data...

Train Monthly Data with Indices Head:


,month,ID,label,VH,VV,blue,green,nir,nira,re1,...,re3,red,swir1,swir2,NDWI,NDVI,MNDWI,NDMI,VH_VV_ratio,VH_VV_diff
0,1,ID_TR_NEW_XVGKFMLNRJ,0,-29.099645,-22.471573,1665,1719,1367,1270,1689,...,1379,1705,1154,1141,0.114064,-0.110026,0.196659,0.084490,1.294954,-6.628071
1,1,ID_TR_NEW_GP8KNSWVP6,0,-19.470574,-10.752340,1579,1740,2245,2231,2060,...,2161,1968,2554,2422,-0.126725,0.065749,-0.189567,-0.064388,1.810822,-8.718234
2,1,ID_TR_NEW_87X3957MVS,1,-20.964854,-8.792675,1850,2345,3664,3470,2935,...,3451,2032,3189,2397,-0.219504,0.286517,-0.152512,0.069313,2.384355,-12.172179
3,1,ID_TR_NEW_T4JMRPKHS3,0,-18.728208,-4.689202,1279,1297,1713,1698,1527,...,1590,1468,1669,1421,-0.138206,0.077020,-0.125421,0.013010,3.993901,-14.039006
4,1,ID_TR_NEW_2CTUQQ8KLU,0,-29.630941,-23.370157,1625,1741,1339,1258,1658,...,1383,1708,1158,1128,0.130519,-0.121103,0.201104,0.072487,1.267897,-6.260785


#### 4.4 Aggregate Features by Location

Now we will aggregate the monthly band values and spectral indices using summary statistics (mean, median, std, min, max, range) over the valid months for each location (`ID`). This step consolidates the time-series data into a fixed-length feature vector per location, making it suitable for traditional machine learning models.

In [ ]:
def aggregate_features(monthly_df, id_col, target_col=None, df_with_valid_months=None):
    """
    Aggregates monthly features using various statistics.
    df_with_valid_months: DataFrame containing ID_COL, num_valid_sar_months, num_valid_optical_months (and target_col if applicable)
                          It should correspond to the IDs in monthly_df.
    """
    # Features to aggregate: original bands + calculated indices
    # Ensure only numeric columns are aggregated
    all_agg_features = [col for col in monthly_df.columns if col not in [id_col, 'month', target_col]]

    # Define aggregation functions
    agg_funcs = ['mean', 'median', 'std', 'min', 'max']

    if not all_agg_features:
        print(f"WARNING: No aggregatable features found in monthly_df for {id_col}. This usually indicates an issue in melt_to_monthly_dataframe or calculate_spectral_indices. Returning ID and num_valid_months only.")
        # If no features to aggregate, just prepare a DataFrame with ID and num_valid_months
        if df_with_valid_months is None:
            if target_col is None: # For test set
                aggregated_df = test_df_processed[[id_col, 'num_valid_sar_months', 'num_valid_optical_months']].drop_duplicates()
            else: # For train set
                aggregated_df = train_df_processed[[id_col, 'num_valid_sar_months', 'num_valid_optical_months', target_col]].drop_duplicates()
        else:
            cols_to_select = [id_col, 'num_valid_sar_months', 'num_valid_optical_months']
            if target_col is not None:
                cols_to_select.append(target_col)
            aggregated_df = df_with_valid_months[cols_to_select].drop_duplicates()
        return aggregated_df

    # Group by ID and aggregate
    aggregated_df = monthly_df.groupby(id_col)[all_agg_features].agg(agg_funcs)
    aggregated_df.columns = [f'{col[0]}_{col[1]}' for col in aggregated_df.columns] # Flatten multi-index columns

    # Add range (max - min) as a custom aggregation
    for feature in all_agg_features:
        if f'{feature}_max' in aggregated_df.columns and f'{feature}_min' in aggregated_df.columns:
            aggregated_df[f'{feature}_range'] = aggregated_df[f'{feature}_max'] - aggregated_df[f'{feature}_min']

    aggregated_df = aggregated_df.reset_index()

    # Merge back number of valid months and target (if training)
    # Use the provided df_with_valid_months for this merge
    if df_with_valid_months is None:
        if target_col is None: # For test set
            original_df_id_num_valid_months = test_df_processed[[id_col, 'num_valid_sar_months', 'num_valid_optical_months']].drop_duplicates()
        else: # For train set
            original_df_id_num_valid_months = train_df_processed[[id_col, 'num_valid_sar_months', 'num_valid_optical_months', target_col]].drop_duplicates()
    else:
        cols_to_select = [id_col, 'num_valid_sar_months', 'num_valid_optical_months']
        if target_col is not None:
            cols_to_select.append(target_col)
        original_df_id_num_valid_months = df_with_valid_months[cols_to_select].drop_duplicates()

    aggregated_df = pd.merge(aggregated_df, original_df_id_num_valid_months, on=id_col, how='left')

    return aggregated_df

print("Aggregating features for train data...")
train_features = aggregate_features(train_monthly_df_indices, ID_COL, TARGET_COL, df_with_valid_months=train_df_processed)
print(f"Train features shape: {train_features.shape}")

print("Aggregating features for test data...")
test_features = aggregate_features(test_monthly_df_indices, ID_COL, df_with_valid_months=test_df_processed)
print(f"Test features shape: {test_features.shape}")

print("\nTrain Features Head:")
display(train_features.head())

print("\nTest Features Head:")
display(test_features.head())

Aggregating features for train data...
Train features shape: (1821, 112)
Aggregating features for test data...
Test features shape: (1030, 111)

Train Features Head:


,ID,VH_mean,VH_median,VH_std,VH_min,VH_max,VV_mean,VV_median,VV_std,VV_min,...,swir2_range,NDWI_range,NDVI_range,MNDWI_range,NDMI_range,VH_VV_ratio_range,VH_VV_diff_range,num_valid_sar_months,num_valid_optical_months,label
0,ID_TR_NEW_22PUHM9YCG,-27.990999,-29.245216,4.978297,-34.411730,-20.775322,-17.732390,-20.196657,4.487905,-22.567680,...,1429,0.401269,0.347787,0.459478,0.152730,0.603706,7.497392,12,12,1
1,ID_TR_NEW_22T3CRYXXR,-24.152243,-24.125484,3.236084,-28.964279,-17.300751,-13.954577,-13.388938,2.083620,-17.207612,...,1558,0.143085,0.167178,0.110639,0.136575,0.956354,11.446587,12,12,0
2,ID_TR_NEW_23BC5PZ4H3,-32.026746,-32.099599,2.120332,-35.708422,-28.761784,-25.555846,-26.111268,1.383130,-27.413006,...,263,0.157199,0.137410,0.187584,0.085763,0.275792,6.301172,12,12,0
3,ID_TR_NEW_23GGHBV5A9,-19.770645,-19.837867,1.096613,-21.350585,-17.539976,-12.631807,-13.333076,1.813971,-14.593050,...,487,0.307945,0.440110,0.117270,0.308780,0.974436,6.557138,12,12,0
4,ID_TR_NEW_24BHKHZ6GK,-30.843903,-30.533366,4.406121,-39.657009,-24.143138,-21.451088,-23.240099,4.049857,-25.958298,...,2290,0.322607,0.352983,0.583097,0.366103,0.669355,11.103478,12,12,0



Test Features Head:


,ID,VH_mean,VH_median,VH_std,VH_min,VH_max,VV_mean,VV_median,VV_std,VV_min,...,swir1_range,swir2_range,NDWI_range,NDVI_range,MNDWI_range,NDMI_range,VH_VV_ratio_range,VH_VV_diff_range,num_valid_sar_months,num_valid_optical_months
0,ID_TS_NEW_01OVVO6U,-30.244819,-29.407085,1.689842,-32.807630,-28.589694,-23.428338,-23.345714,0.501748,-24.098688,...,324.0,259.0,0.185641,0.092548,0.119338,0.222073,0.220899,4.751639,6,6
1,ID_TS_NEW_01T3I13F,-26.669745,-26.494660,1.798767,-29.018415,-24.671244,-18.444556,-18.864129,4.458982,-23.027250,...,1147.0,616.0,0.221425,0.148562,0.255917,0.047691,0.732612,7.921197,4,4
2,ID_TS_NEW_02SGGVQX,-25.605754,-25.783779,1.631034,-26.995351,-22.853247,-18.275647,-18.708474,3.291946,-22.381989,...,1052.0,722.0,0.240386,0.252527,0.349219,0.154377,0.462325,5.716637,5,5
3,ID_TS_NEW_030K8WXY,-28.834774,-27.841509,3.044433,-33.187780,-25.740172,-17.478971,-18.308199,2.091285,-19.351680,...,2503.0,2128.0,0.188793,0.204343,0.618035,0.472957,0.600802,9.056198,5,5
4,ID_TS_NEW_05H6TLM4,-31.634558,-31.332035,1.730390,-33.992820,-29.226948,-21.917357,-21.308735,1.163898,-23.259771,...,345.0,259.0,0.050070,0.059330,0.029097,0.078647,0.352747,6.856882,5,3


#### 4.5 Handle Remaining Missing Values

After aggregation, some features might still contain `NaN` values (e.g., if a location had zero valid optical months, all optical-derived indices would be `NaN`). We need to explicitly handle these remaining missing values before model training. For simplicity and robustness, we will impute these with the median value of each column.

In [ ]:
print("Handling remaining missing values...")

# Identify feature columns for the aggregated dataframes
final_feature_cols = [col for col in train_features.columns if col not in [ID_COL, TARGET_COL]]

# Impute missing values with the median for each feature column
# We fit the imputer on the training data and transform both train and test.
for col in final_feature_cols:
    if train_features[col].isnull().any():
        median_val = train_features[col].median()
        train_features[col] = train_features[col].fillna(median_val)
        test_features[col] = test_features[col].fillna(median_val)

print("Missing values after imputation in train features:")
print(train_features[final_feature_cols].isnull().sum().sum())
print("Missing values after imputation in test features:")
print(test_features[final_feature_cols].isnull().sum().sum())

# Prepare X and y for modeling
X = train_features.drop(columns=[ID_COL, TARGET_COL])
y = train_features[TARGET_COL]
X_test = test_features.drop(columns=[ID_COL])

print(f"\nShape of X: {X.shape}, y: {y.shape}")
print(f"Shape of X_test: {X_test.shape}")

Handling remaining missing values...
Missing values after imputation in train features:
0
Missing values after imputation in test features:
0

Shape of X: (1821, 110), y: (1821,)
Shape of X_test: (1030, 110)


## 5. Model Training and Validation

We will train several gradient-boosted tree models (LightGBM, XGBoost, CatBoost) and a Logistic Regression baseline. A stratified k-fold cross-validation strategy will be used, with a special consideration for simulating the train/test domain shift by randomly masking out 4-6 consecutive months from training data during CV.

### 5.1 Define Evaluation Metrics

The competition uses a weighted average of F1-score (60%) and ROC-AUC (40%). We'll define a function to calculate this custom metric.

In [ ]:
def custom_zindi_metric(y_true, y_pred_proba, y_pred_binary):
    f1 = f1_score(y_true, y_pred_binary)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    weighted_score = 0.6 * f1 + 0.4 * roc_auc
    return f1, roc_auc, weighted_score

print("Custom Zindi metric function defined.")

Custom Zindi metric function defined.


### 5.2 Cross-Validation with Domain Shift Simulation

To better reflect the test set conditions (only 4-6 consecutive months of real data), during cross-validation, we will randomly mask training rows to simulate this data sparsity. This helps the model learn features that are robust to missing monthly data rather than memorizing seasonal patterns.

In [ ]:
def simulate_domain_shift_masking(df_processed, feature_cols, sar_bands, optical_bands, num_months_to_keep=None):
    """
    Randomly masks all but `num_months_to_keep` consecutive months in a row.
    If num_months_to_keep is None, it randomly picks between 4, 5, or 6 months.
    Returns a new DataFrame with masked values replaced by NaN.
    """
    df_masked = df_processed.copy()
    np.random.seed(SEED) # Ensure reproducibility of masking

    num_total_months = 12

    for index, row in tqdm(df_masked.iterrows(), total=len(df_masked), desc="Simulating domain shift"):
        if num_months_to_keep is None:
            # Randomly select 4, 5, or 6 months to keep
            k = np.random.choice([4, 5, 6])
        else:
            k = num_months_to_keep

        # Choose a random starting month for a consecutive window of k months
        # The window can span across the year boundary (e.g., month 11, 12, 1, 2 for k=4)
        start_month_offset = np.random.randint(0, num_total_months)
        months_to_keep = [(start_month_offset + i) % num_total_months + 1 for i in range(k)]

        # Identify months to mask (all months not in months_to_keep)
        months_to_mask = [m for m in range(1, num_total_months + 1) if m not in months_to_keep]

        for month_idx in months_to_mask:
            # Get all columns for this month and mask them
            cols_to_mask = []
            for band in ALL_BANDS: # Use ALL_BANDS to get all feature columns for that month
                col_name = f'{band}_{month_idx:02d}'
                if col_name in feature_cols: # Check if the column actually exists in the dataframe
                    cols_to_mask.append(col_name)
            df_masked.loc[index, cols_to_mask] = np.nan

    # After masking, re-run valid month identification and re-aggregate features
    # Note: this will be done inside the CV loop for each fold's training data.
    return df_masked


print("Domain shift simulation function defined.")

Domain shift simulation function defined.


### 5.3 Cross-Validation Function

This function orchestrates cross-validation, including domain shift simulation, model training, prediction, and metric calculation.

In [ ]:
def run_cross_validation(model_class, model_params, X, y, X_test,
                         train_df_full, test_df_full, feature_cols_original_df,
                         sar_bands, optical_bands, all_bands,
                         n_splits=5,
                         simulate_shift=False,
                         model_name='Model'):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    oof_preds_proba = np.zeros(len(X))
    test_preds_proba = np.zeros(len(X_test))
    feature_importances_folds = []

    print(f"\nStarting {model_name} Cross-Validation...")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")

        train_fold_original_ids = train_df_full.iloc[train_idx][ID_COL].values
        val_fold_original_ids = train_df_full.iloc[val_idx][ID_COL].values

        X_train_fold_processed = train_df_full[train_df_full[ID_COL].isin(train_fold_original_ids)].copy().reset_index(drop=True)
        y_train_fold_original = y.iloc[train_idx].reset_index(drop=True)

        X_val_fold = X.iloc[val_idx]
        y_val_fold = y.iloc[val_idx]

        if simulate_shift:
            print(f"  Simulating domain shift for training data in Fold {fold+1}...")

            train_df_processed_fold_masked = simulate_domain_shift_masking(
                X_train_fold_processed, feature_cols_original_df, sar_bands, optical_bands
            )

            train_valid_sar_months_masked, train_valid_optical_months_masked = identify_valid_months(
                train_df_processed_fold_masked, sar_bands, optical_bands
            )

            train_df_processed_fold_masked['num_valid_sar_months'] = train_valid_sar_months_masked.sum(axis=1)
            train_df_processed_fold_masked['num_valid_optical_months'] = train_valid_optical_months_masked.sum(axis=1)

            train_monthly_df_masked = melt_to_monthly_dataframe(
                train_df_processed_fold_masked, train_valid_sar_months_masked, all_bands, ID_COL, TARGET_COL
            )
            train_monthly_df_indices_masked = calculate_spectral_indices(train_monthly_df_masked)

            X_train_fold_reaggregated_with_labels_and_ids = aggregate_features(
                train_monthly_df_indices_masked, ID_COL, TARGET_COL,
                df_with_valid_months=train_df_processed_fold_masked
            )

            X_train_fold_reaggregated_with_labels_and_ids = X_train_fold_reaggregated_with_labels_and_ids[
                X_train_fold_reaggregated_with_labels_and_ids[ID_COL].isin(train_fold_original_ids)
            ]
            y_train_fold = X_train_fold_reaggregated_with_labels_and_ids[TARGET_COL]

            X_train_fold_temp = X_train_fold_reaggregated_with_labels_and_ids.drop(columns=[ID_COL, TARGET_COL])

            missing_cols_reagg = set(X.columns) - set(X_train_fold_temp.columns)
            for c in missing_cols_reagg:
                X_train_fold_temp[c] = np.nan
            X_train_fold_temp = X_train_fold_temp[X.columns]

            for col_name in X_train_fold_temp.columns:
                if X_train_fold_temp[col_name].isnull().any():
                    median_val = X_train_fold_temp[col_name].median()
                    X_train_fold_temp[col_name] = X_train_fold_temp[col_name].fillna(median_val)

            X_train_fold = X_train_fold_temp

        else:
            X_train_fold = X.iloc[train_idx]
            y_train_fold = y.iloc[train_idx]

        model = model_class(**model_params)

        if model_name == 'CatBoost':
            # early_stopping_rounds already set in model_params (constructor);
            # just supply eval_set here so it actually takes effect.
            model.fit(X_train_fold, y_train_fold,
                      eval_set=(X_val_fold, y_val_fold),
                      verbose=0)
        else:
            model.fit(X_train_fold, y_train_fold)

        oof_preds_proba[val_idx] = model.predict_proba(X_val_fold)[:, 1]
        test_preds_proba += model.predict_proba(X_test)[:, 1] / n_splits

        if hasattr(model, 'feature_importances_'):
            feature_importances_folds.append(pd.Series(model.feature_importances_, index=X.columns))
        elif hasattr(model, 'coef_') and model_name == 'Logistic Regression':
            feature_importances_folds.append(pd.Series(model.coef_[0], index=X.columns))

    oof_preds_binary = (oof_preds_proba > 0.5).astype(int)
    f1_oof, roc_auc_oof, weighted_score_oof = custom_zindi_metric(y, oof_preds_proba, oof_preds_binary)

    print(f"\n{model_name} OOF Results:")
    print(f"  F1 Score: {f1_oof:.4f}")
    print(f"  ROC AUC: {roc_auc_oof:.4f}")
    print(f"  Weighted Zindi Score (0.6*F1 + 0.4*AUC): {weighted_score_oof:.4f}")

    feature_importances_df = pd.DataFrame(feature_importances_folds).T

    return oof_preds_proba, test_preds_proba, feature_importances_df


### 5.4 Model Training

We will train a Logistic Regression baseline and three gradient-boosted tree models: LightGBM, XGBoost, and CatBoost. Each model will undergo cross-validation, and their out-of-fold predictions will be collected for potential ensembling.

In [ ]:
all_oof_preds_proba = {}
all_test_preds_proba = {}
all_feature_importances = {}

original_df_feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]

# --- 1. Logistic Regression (Baseline) ---
print("\n--- Training Logistic Regression ---")
log_reg_params = {
    'solver': 'liblinear',
    'penalty': 'l1',
    'C': 0.1,
    'random_state': SEED,
    'class_weight': 'balanced'
}
log_reg_oof, log_reg_test, log_reg_fi = run_cross_validation(
    LogisticRegression, log_reg_params, X, y, X_test,
    train_df_processed, test_df_processed, original_df_feature_cols, SAR_BANDS, OPTICAL_BANDS, ALL_BANDS,
    simulate_shift=True, model_name='Logistic Regression'
)
all_oof_preds_proba['log_reg'] = log_reg_oof
all_test_preds_proba['log_reg'] = log_reg_test
all_feature_importances['log_reg'] = log_reg_fi

# --- 2. LightGBM ---
print("\n--- Training LightGBM ---")
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'seed': SEED,
    'n_jobs': -1,
    'verbose': -1,
    'colsample_bytree': 0.7,
    'subsample': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'is_unbalance': True
}
lgb_oof, lgb_test, lgb_fi = run_cross_validation(
    lgb.LGBMClassifier, lgb_params, X, y, X_test,
    train_df_processed, test_df_processed, original_df_feature_cols, SAR_BANDS, OPTICAL_BANDS, ALL_BANDS,
    simulate_shift=True, model_name='LightGBM'
)
all_oof_preds_proba['lgbm'] = lgb_oof
all_test_preds_proba['lgbm'] = lgb_test
all_feature_importances['lgbm'] = lgb_fi

# --- 3. XGBoost ---
print("\n--- Training XGBoost ---")
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    # 'use_label_encoder' removed -- not supported in xgboost>=2.0
    'seed': SEED,
    'n_jobs': -1,
    'tree_method': 'hist',
    'scale_pos_weight': len(y[y==0]) / len(y[y==1])
}
xgb_oof, xgb_test, xgb_fi = run_cross_validation(
    xgb.XGBClassifier, xgb_params, X, y, X_test,
    train_df_processed, test_df_processed, original_df_feature_cols, SAR_BANDS, OPTICAL_BANDS, ALL_BANDS,
    simulate_shift=True, model_name='XGBoost'
)
all_oof_preds_proba['xgb'] = xgb_oof
all_test_preds_proba['xgb'] = xgb_test
all_feature_importances['xgb'] = xgb_fi

# --- 4. CatBoost ---
print("\n--- Training CatBoost ---")
cb_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'verbose': 0,
    'early_stopping_rounds': 50,
    'class_weights': [1, len(y[y==0]) / len(y[y==1])]
}
cb_oof, cb_test, cb_fi = run_cross_validation(
    cb.CatBoostClassifier, cb_params, X, y, X_test,
    train_df_processed, test_df_processed, original_df_feature_cols, SAR_BANDS, OPTICAL_BANDS, ALL_BANDS,
    simulate_shift=True, model_name='CatBoost'
)
all_oof_preds_proba['catboost'] = cb_oof
all_test_preds_proba['catboost'] = cb_test
all_feature_importances['catboost'] = cb_fi



--- Training Logistic Regression ---

Starting Logistic Regression Cross-Validation...

--- Fold 1/5 ---
  Simulating domain shift for training data in Fold 1...


Simulating domain shift:   0%|          | 0/1456 [00:00<?, ?it/s]


--- Fold 2/5 ---
  Simulating domain shift for training data in Fold 2...


Simulating domain shift:   0%|          | 0/1457 [00:00<?, ?it/s]


--- Fold 3/5 ---
  Simulating domain shift for training data in Fold 3...


Simulating domain shift:   0%|          | 0/1457 [00:00<?, ?it/s]


--- Fold 4/5 ---
  Simulating domain shift for training data in Fold 4...


Simulating domain shift:   0%|          | 0/1457 [00:00<?, ?it/s]


--- Fold 5/5 ---
  Simulating domain shift for training data in Fold 5...


Simulating domain shift:   0%|          | 0/1457 [00:00<?, ?it/s]

### 5.5 Model Ensembling/Blending

We will blend the predictions from the trained models by simply averaging their probabilities. This often leads to more robust predictions than a single model.

In [ ]:
print("\n--- Ensembling Models ---")

# Average OOF probabilities
ensembled_oof_proba = np.mean(list(all_oof_preds_proba.values()), axis=0)
ensembled_oof_binary = (ensembled_oof_proba > 0.5).astype(int)

# Calculate ensembled OOF metrics
f1_ensembled, roc_auc_ensembled, weighted_score_ensembled = custom_zindi_metric(y, ensembled_oof_proba, ensembled_oof_binary)

print(f"Ensembled OOF Results:")
print(f"  F1 Score: {f1_ensembled:.4f}")
print(f"  ROC AUC: {roc_auc_ensembled:.4f}")
print(f"  Weighted Zindi Score (0.6*F1 + 0.4*AUC): {weighted_score_ensembled:.4f}")

# Average test probabilities
ensembled_test_proba = np.mean(list(all_test_preds_proba.values()), axis=0)

print("Ensembling complete.")


--- Ensembling Models ---
Ensembled OOF Results:
  F1 Score: 0.9530
  ROC AUC: 0.9865
  Weighted Zindi Score (0.6*F1 + 0.4*AUC): 0.9664
Ensembling complete.


## 6. Feature Importance

Understanding which features contribute most to the model's predictions is crucial for interpretability and further insights. We'll examine the feature importances from our gradient-boosted models.

In [ ]:
print("\n--- Analyzing Feature Importances ---")

# Combine feature importances from tree-based models
# For simplicity, we'll average importances across models (excluding Logistic Regression as it has coefficients, not feature_importances)

combined_fi = pd.DataFrame(index=X.columns)

# Ensure all feature importance DFs are not empty before averaging
filtered_fi = {name: df for name, df in all_feature_importances.items() if not df.empty and name != 'log_reg'}

if filtered_fi:
    for model_name, fi_df in filtered_fi.items():
        combined_fi[model_name] = fi_df.mean(axis=1)

    if not combined_fi.empty:
        combined_fi['average_importance'] = combined_fi.mean(axis=1)
        top_features = combined_fi.sort_values(by='average_importance', ascending=False).head(20)

        print("\nTop 20 Most Important Features (Average across tree models):")
        display(top_features)

        # Discussion point: Which band/index/statistic matters most?
        # Extract original band names and aggregation statistics from the top features
        def parse_feature_name(feature_name):
            parts = feature_name.split('_')
            # Heuristic to distinguish band from statistic
            # Assuming statistics are usually at the end and are shorter (mean, std, min, max, range, median)
            possible_stats = ['mean', 'median', 'std', 'min', 'max', 'range']
            if parts and parts[-1].lower() in possible_stats:
                agg_stat = parts[-1]
                band_name = '_'.join(parts[:-1])
            else:
                # If no clear stat, assume the whole name is the band/index
                band_name = '_'.join(parts)
                agg_stat = 'overall'
            return band_name, agg_stat

        parsed_top_features = []
        for feature_name, importance in top_features['average_importance'].items():
            band, stat = parse_feature_name(feature_name)
            parsed_top_features.append({'Feature': feature_name, 'Band/Index': band, 'Statistic': stat, 'Importance': importance})

        parsed_top_features_df = pd.DataFrame(parsed_top_features)
        print("\nParsed Top Features:")
        display(parsed_top_features_df)

        # Further analysis could group by band/index and statistic
        band_importance = parsed_top_features_df.groupby('Band/Index')['Importance'].sum().sort_values(ascending=False)
        stat_importance = parsed_top_features_df.groupby('Statistic')['Importance'].sum().sort_values(ascending=False)

        print("\nTop Band/Index Types by Importance:")
        print(band_importance.head())
        print("\nTop Aggregation Statistics by Importance:")
        print(stat_importance.head())

        print("\nBrief discussion of insights from feature importances:\n")
        print("Based on the average feature importances from LightGBM, XGBoost, and CatBoost, we can observe:")
        print(f"- The most influential features often involve specific bands or derived spectral indices like {band_importance.index[0]} and {band_importance.index[1]}.")
        print(f"- Summary statistics such as '{stat_importance.index[0]}' and '{stat_importance.index[1]}' frequently appear among the top features, indicating that temporal variability and central tendencies are important.")
        print("This suggests that the models effectively capture the characteristic spectral and temporal signatures of aquaculture ponds.")
else:
    print("No feature importances to display from tree models as `filtered_fi` is empty.")


--- Analyzing Feature Importances ---
No feature importances to display from tree models as `filtered_fi` is empty.


## 7. Final Training & Prediction

To create the final submission, we will retrain the best performing (or ensembled) model on the full training dataset and make predictions on the test set.

In [ ]:
print("\n--- Final Training and Prediction ---")

# For the final submission, we will use the ensembled test probabilities.
final_test_predictions_proba = ensembled_test_proba
final_test_predictions_binary = (final_test_predictions_proba > 0.5).astype(int)

print("Final predictions generated.")


--- Final Training and Prediction ---
Final predictions generated.


## 8. Submission Export & Validation

Finally, we will create the `submission.csv` file in the format required by the competition and perform a validation check to ensure correctness before uploading.

In [ ]:
# Create submission DataFrame
submission_df = pd.DataFrame({
    SUBMISSION_ID_COL: test_df[ID_COL],
    SUBMISSION_TARGET_COL: final_test_predictions_binary,
    SUBMISSION_PROB_COL: final_test_predictions_proba
})

# Ensure the ID order matches the sample submission (important for Zindi challenges)
submission_df = pd.merge(sample_submission_df[[SUBMISSION_ID_COL]], submission_df, on=SUBMISSION_ID_COL, how='left')

# Save the submission file
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Submission file '{submission_filename}' created successfully.")

# --- Submission File Validation ---
print("\n--- Validating Submission File ---")

# Load the generated submission file
validated_submission_df = pd.read_csv(submission_filename)

# 1. Check shape
expected_shape = sample_submission_df.shape
actual_shape = validated_submission_df.shape
assert actual_shape == expected_shape, f"Shape mismatch: Expected {expected_shape}, got {actual_shape}"
print(f"- Shape is correct: {actual_shape}")

# 2. Check columns
expected_cols = sample_submission_df.columns.tolist()
actual_cols = validated_submission_df.columns.tolist()
assert actual_cols == expected_cols, f"Column mismatch: Expected {expected_cols}, got {actual_cols}"
print(f"- Columns are correct: {actual_cols}")

# 3. Check for NaNs
assert validated_submission_df.isnull().sum().sum() == 0, "NaN values found in submission file"
print("- No NaN values found.")

# 4. Check IDs match SampleSubmission exactly (order and values)
assert validated_submission_df[SUBMISSION_ID_COL].equals(sample_submission_df[SUBMISSION_ID_COL]), "ID column mismatch or order incorrect."
print("- ID column matches sample submission exactly.")

# 5. Check Target column is binary (0/1)
assert validated_submission_df[SUBMISSION_TARGET_COL].isin([0, 1]).all(), "TargetF1 column contains non-binary values."
print("- TargetF1 column is binary (0/1).")
print("\nSubmission file validation successful! Ready for upload.")
from google.colab import files

# Download the submission file
files.download('submission.csv')

Submission file 'submission.csv' created successfully.

--- Validating Submission File ---
- Shape is correct: (1030, 3)
- Columns are correct: ['ID', 'TargetF1', 'TargetRAUC']
- No NaN values found.
- ID column matches sample submission exactly.
- TargetF1 column is binary (0/1).

Submission file validation successful! Ready for upload.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>